# Exercise 04: Merge and Heatmap — Demographic Bias Audit
# Cluster 4 - Pandas and Visualization
# Framework: EU AI Act Article 10 (Data Governance), NIST AI RMF Map + Measure
# Case Study: Recruitment AI 🟢

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Dataset 1: Hiring decisions
applicants = pd.DataFrame({
    'applicant_id': range(1, 51),
    'test_score': np.random.randint(50, 100, 50),
    'hired': np.random.choice([0, 1], 50, p=[0.6, 0.4])
})

In [ ]:
applicants.head(10)

In [ ]:
# Dataset 2: Demographic information
demographics = pd.DataFrame({
    'applicant_id': range(1, 51),
    'gender': np.random.choice(['Woman', 'Man'], 50, p=[0.5, 0.5]),
    'age_group': np.random.choice(['20-30', '31-40', '41-50'], 50)
})
demographics.head(10)

In [ ]:
# Merge 1: Combine hiring decisions with demographic data
audit_df = pd.merge(applicants, demographics, on='applicant_id', how='inner')
print(audit_df.shape)
audit_df.head()

In [ ]:
# Dataset 3: Interview panel scores (only 40 of 50 applicants were interviewed)
panel_scores = pd.DataFrame({
    'applicant_id': np.random.choice(range(1, 51), 40, replace=False),
    'panel_score': np.random.randint(40, 100, 40)
})

In [ ]:
panel_scores.head(10) 


In [ ]:
panel_scores.shape

In [ ]:
# Merge 2: Add panel scores to audit dataframe
# Left join — keep all 50 applicants, even those without panel scores
audit_full = pd.merge(audit_df, panel_scores, on='applicant_id', how='left')
print(audit_full.shape)
audit_full.isnull().sum()

In [ ]:
audit_full.head()

In [ ]:
# Who didn't make it to interview?
missing_panel = audit_full[audit_full['panel_score'].isnull()]
missing_panel['gender'].value_counts()

In [ ]:
# Create a contingency table: hiring rates by gender and age group
pivot_table = audit_full.groupby(['gender', 'age_group'])['hired'].agg(['sum', 'count'])
pivot_table['hiring_rate'] = (pivot_table['sum'] / pivot_table['count'] * 100).round(1)
pivot_table_display = pivot_table[['hiring_rate']].unstack()
print(pivot_table_display)

In [ ]:
# Seaborn heatmap: hiring rates by gender and age group
plt.figure(figsize=(8, 4))
sns.heatmap(pivot_table_display, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn',
            linewidths=0.5)
plt.title('Hiring Rate (%) by Gender and Age Group\nRecruitment AI Bias Audit')
plt.ylabel('Gender')
plt.xlabel('Age Group')
plt.tight_layout()
plt.show()

## Governance Reflection: Heatmap Analysis

Looking at this heatmap, I see a clear age-based bias pattern for women but not for men. Women aged 31-40 have the highest hiring rate at 60%, but women aged 41-50 drop dramatically to 37.5%. For men, the pattern is the opposite — hiring rates improve with age (23.1% → 40.0% → 42.9%). 

This suggests the system may have age bias that affects women and men differently. Older women are being filtered out, while older men are being preferred. If I were presenting this to a hiring committee, I would ask: why does the system treat experienced women differently than experienced men? This could indicate a data or training issue in how the hiring algorithm evaluates candidates.

This connects to EU AI Act Article 10 (Data Governance) — we need to audit what data the system was trained on, and whether it contains historical biases against older women.